# Objective verification — legacy vs current, ResNet-50 / CIFAR-10

Does the ORIGINAL contrastive design beat the refactored one? Two arms, BaCP + magnitude, sparsities 0.95 / 0.97 / 0.99, seed 1, shared dense checkpoint, identical protocol. **Exactly two config keys differ:**

| | legacy arm | control arm |
|---|---|---|
| `contrastive_mode` | `legacy` — SupCon + NT-Xent over a 2B x 2B `cat([student, teacher])` matrix (SimCLR-style, all pairs) | `cap` — one CAP Eq. 1 functional, rectangular B x N |
| `proj_mode` | `current` — per-model projection heads, student's trainable | `tied_frozen` — one frozen head shared by student and teachers |

Everything else is held: tau 0.15, 60 epochs, delta_T 88, cubic ramp to 80% then 20% recovery, classifier pruned, SGD 0.1 -> AdamW 1e-4 x 50, lambdas 0.25 each, same data and eval.

Both arms are compared against the SAME I.P. baseline, so the gain over I.P. is the quantity of interest.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Sanity check — run this and read it before training

Verifies (1) the protocol, (2) what will run vs what already exists, and that the ablation differs from its control in exactly the two keys under test, and (3) the dataset and which split the accuracy comes from.

In [ ]:
LEGACY = dict(contrastive_mode='legacy', proj_mode='current')
SPARSITIES = (0.95, 0.97, 0.99, 0.999)

legacy_cells = [nb.make_cell('resnet50', 'bacp', seed=1, pruner='magnitude',
                             sparsity=s, variant='legacy', **LEGACY)
                for s in SPARSITIES]
control_cells = [nb.make_cell('resnet50', 'bacp', seed=1, pruner='magnitude',
                              sparsity=s)
                 for s in SPARSITIES]

# the ablation is checked against its OWN sparsity's control
assert nb.sanity_check(legacy_cells, control=control_cells[0]), 'legacy arm failed sanity'
assert nb.sanity_check(control_cells), 'control arm failed sanity'

## Run both arms

Anything already recorded is skipped. tqdm streams per-batch progress; 24 dataloader workers.

In [ ]:
for cell in legacy_cells + control_cells:
    nb.run(cell, gpu=0)

## Verdict

In [ ]:
nb.verification_table()